# 6.5 換 tokenizer 後如何比較模型？


同一段原文，甲拆成十token、平均代價0.6，乙拆成五token、平均代價1.0。只看平均會說甲較好，可是甲總代價6、乙只有5。單位改變後，「每token」不是同樣的一份文字，因此比較應回到固定原文，量整段總負對數機率，再用同一個原文長度單位除。

[1.8](https://birdhackor.github.io/tiny-perceptron-vlm/1.8.html) 的每位置代價是正確token機率的負自然對數，把整段加總叫negative log likelihood（負對數似然，NLL）。根據 [W.4](https://birdhackor.github.io/tiny-perceptron-vlm/first-steps.html#W.4) 的對數加法，它對應這段文字在給定前文下的聯合概率。再除以原文UTF-8的byte數量、除以 `log(2)` 換成bit，得到bits per byte，簡稱BPB，表示每原始byte的代價。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import math

raw = "貓🙂"
bytes_count = len(raw.encode("utf-8"))
for name, total_nll in [("示例A", 5.0), ("示例B", 4.0)]:
    bpb = total_nll / (bytes_count * math.log(2))
    print(name, "原文bytes", bytes_count, "BPB", bpb)

輸入同一原文貓與🙂，共三加四等於七bytes，長度差的來源見 [6.1](https://birdhackor.github.io/tiny-perceptron-vlm/6.1.html)。總NLL5與4是手工例子，尚非模型量測。輸出A約1.0305、B約0.8244，因為除數同為 `7×0.6931`，B較低表示這份原文總負對數代價較小。即使兩者token數不同，也回到了相同分母。

真正測量時，先用各自tokenizer編碼同一份驗證文件，再交給與它匹配的模型預測。有效目標位置是「有真實的下一文字單位，而且這次要替它計代價」的位置；只用來提供前文的內容，或為湊等長而額外填入的記號，不算本次的原文目標。把這些目標的負log機率加總，才是本次的總NLL。不能拿甲的ID丟進乙的模型，也不能只拿平均loss、沒有目標數量就推BPB。

長文件放不進模型時，會截成有限長度的片段，叫窗口，詳見[6.8](https://birdhackor.github.io/tiny-perceptron-vlm/6.8.html)。例如原文有四個token，依次稱為A、B、C、D；若A先當已知前文，本次要評的是接著猜B、C、D。切成`[A,B,C]`與`[B,C,D]`兩個重疊窗口時，可在第一個計B、C的代價；第二個的B、C只提供前文，計D即可。若第二個又計C，同一原文目標就被算了兩次。兩個模型都要採用一致的計分約定，並記錄各目標能讀到多少前文。文件開始的已知條件、結束標記是否計入NLL，也要先說清楚，才能比較同一份原文的相同事件。

BOS是開始標記、EOS是結束標記，後面 [6.6](https://birdhackor.github.io/tiny-perceptron-vlm/6.6.html) 會給專用ID；這裡先理解它們屬於結構，不是原文bytes。若把EOS代價計入分子，卻用原文bytes分母，這可以作為一種明確的評估契約，但兩方法都要一致，不可只為一方略過難預測的結尾。

BPB讓拆分單位較可比，卻也不是回答品質的唯一指標。模型善於預測原文，並不自動善於自由回答問題，[5.8](https://birdhackor.github.io/tiny-perceptron-vlm/5.8.html) 已區分這兩者。對照應同時保留任務表現、上下文預算與總資源；byte長度統一不表示兩者訓練成本也相同。

練習只把A與B的total_nll都乘2，先預測BPB都翻倍、排序不變，再執行核對。然後只把raw換成兩倍原文 `"貓🙂貓🙂"`，保留同一總NLL，數值會減半；這一步是在演示分母作用，真實模型換文本必須重測NLL，不能用舊數字當新結果。
